# ldd 依存チェック



In [ ]:

import subprocess
import glob
import re

dist = "/usr/local/lib/python3.13/dist-packages"
libs = glob.glob(f"{dist}/**/*.so", recursive=True)[:200]
print(f"対象 .so: {len(libs)} 件")

found_bad = []
found_rpath = []
gpu_deps = []

for lib in libs:
    try:
        r = subprocess.run(["ldd", lib], capture_output=True, text=True, timeout=5)
        out = (r.stdout or "") + (r.stderr or "")
    except Exception as e:
        out = str(e)

    bad = re.findall(r"not found", out, re.IGNORECASE)
    rpath = re.findall(r"(?:RPATH|RUNPATH)=(.+?)\n", out)

    if bad:
        found_bad.append((lib, bad))
    if rpath:
        found_rpath.append((lib, rpath))
    if any(dep in out.lower() for dep in ["libcud", "libnvidia"]):
        gpu_deps.append(lib)

print(f"\n=== not found: {len(found_bad)} ===")
for lib, msgs in found_bad[:10]:
    print(f"  {lib}: {msgs}")

print(f"\n=== RPATH/RUNPATH: {len(found_rpath)} ===")
for lib, paths in found_rpath[:5]:
    print(f"  {lib}: {paths}")

print(f"\n=== GPU lib 依存: {len(gpu_deps)} ===")
for lib in gpu_deps[:10]:
    print(f"  {lib}")

print("\n=== ldd 完了 ===")
